# 🚀 Stock Bot AutoEncoder Training on Google Colab

이 노트북은 한국 주식 시장 스캘핑을 위한 AutoEncoder 기반 임베딩 모델을 Google Colab에서 훈련합니다.

## 📋 Overview
- **목적**: 시계열 데이터를 128차원 임베딩으로 압축
- **모델**: Masked AutoEncoder with Transformer
- **데이터**: 전처리된 HDF5 배치 파일
- **훈련 시간**: 약 4-6시간 (GPU 사용시)

## 🔧 Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install h5py numpy pandas tqdm matplotlib seaborn

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup directories
import os
os.chdir('/content')
!mkdir -p models logs data

# Data path
DATA_PATH = '/content/drive/MyDrive/models/stockbot/pre_training_data'

# Check data
if os.path.exists(DATA_PATH):
    print(f"✅ Data found at: {DATA_PATH}")
    months = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
    print(f"Available months: {sorted(months)}")
else:
    print(f"❌ Data not found at: {DATA_PATH}")

## Github

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os
from google.colab import userdata

# 🔑 아이콘 클릭 → Add new secret
# Name: GITHUB_TOKEN
# Value: your_personal_access_token
try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token for authentication")
except:
    print("⚠️ GITHUB_TOKEN not found. Using public clone (may have rate limits)")
    use_token = False

# 이미 클론되어 있으면 스킵
if not os.path.exists('/content/stock-bot2'):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git /content/stock-bot2
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git /content/stock-bot2
else:
    print("📁 Repository already exists, updating...")
    %cd /content/stock-bot2
    !git fetch origin && git pull

# 작업 디렉토리 변경
%cd /content/stock-bot2

# Python path 추가
import sys
sys.path.append('/content/stock-bot2')

print("✅ Repository cloned and ready!")
print(f"📂 Current directory: {os.getcwd()}")

## 📚 Import Libraries and Models

In [ ]:
# Import standard libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import json
import random
import math
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import project modules
try:
    from ai_trader.embedding.autoencoder_model import MaskedAutoEncoder
    from ai_trader.embedding.data import PreprocessedDataset, BatchCollator
    print("✅ Successfully imported project modules")
except ImportError as e:
    print(f"❌ Failed to import project modules: {e}")
    print("📝 Will define models inline instead")
    
    # Fallback: Define models inline if import fails
    exec(open('/content/stock-bot2/ai_trader/embedding/autoencoder_model.py').read())
    exec(open('/content/stock-bot2/ai_trader/embedding/data.py').read())

## ⚙️ Training Configuration

In [ ]:
# Configuration
CONFIG = {
    'embedding_dim': 128,
    'hidden_dim': 256,
    'num_layers': 3,
    'dropout': 0.1,
    'mask_ratio': 0.15,
    
    'batch_size': 4,
    'max_sequences': 2000,
    'max_epochs': 15,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    
    'max_batches_per_month': 30,
    'train_months': ['2024_09', '2024_10', '2024_11'],
    'val_months': ['2024_12'],
}

# GPU memory adjustment
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️ GPU Memory: {gpu_memory:.1f} GB")
    
    if gpu_memory < 12:
        CONFIG['max_sequences'] = 1000
        CONFIG['batch_size'] = 2
        CONFIG['max_batches_per_month'] = 20
        print("⚠️ Adjusted config for limited GPU memory")
    elif gpu_memory > 25:
        CONFIG['max_sequences'] = 4000
        CONFIG['batch_size'] = 8
        CONFIG['max_batches_per_month'] = 100
        print("🚀 Adjusted config for high-end GPU")

print("📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 📁 Data Setup

In [ ]:
# Create datasets
print("📊 Creating datasets...")

train_dataset = PreprocessedDataset(
    DATA_PATH, 
    CONFIG['train_months'], 
    CONFIG['max_batches_per_month']
)

val_dataset = PreprocessedDataset(
    DATA_PATH, 
    CONFIG['val_months'], 
    CONFIG['max_batches_per_month']
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    collate_fn=BatchCollator(CONFIG['max_sequences']),
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    collate_fn=BatchCollator(CONFIG['max_sequences']),
    num_workers=2,
    pin_memory=True
)

print(f"📈 Training batches: {len(train_loader)}")
print(f"📉 Validation batches: {len(val_loader)}")
print(f"🔢 Data shape: {train_dataset.seq_len} x {train_dataset.num_features}")

## 🧠 Model Setup

In [ ]:
# Create model
print("🧠 Creating model...")

model = MaskedAutoEncoder(
    input_dim=train_dataset.num_features,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    seq_len=train_dataset.seq_len,
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    mask_ratio=CONFIG['mask_ratio']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB")

# Create optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.95)
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['max_epochs'],
    eta_min=CONFIG['learning_rate'] * 0.01
)

print("✅ Model, optimizer, and scheduler created")

## 🚀 Training

In [ ]:
# Training loop
print("🚀 Starting training...")

history = {'train_loss': [], 'val_loss': [], 'learning_rate': []}
best_val_loss = float('inf')
start_time = time.time()

for epoch in range(CONFIG['max_epochs']):
    epoch_start = time.time()
    
    # Training
    model.train()
    train_loss = 0.0
    train_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['max_epochs']}")
    for batch in pbar:
        batch = batch.to(device)
        
        optimizer.zero_grad()
        
        reconstruction, embedding, mask = model(batch)
        
        # Reconstruction loss (only on masked tokens)
        recon_loss = F.mse_loss(reconstruction[mask], batch[mask])
        
        # Regularization loss
        reg_loss = 0.01 * torch.norm(embedding, dim=1).mean()
        
        total_loss = recon_loss + reg_loss
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += total_loss.item()
        train_batches += 1
        
        pbar.set_postfix({
            'loss': f"{total_loss.item():.4f}",
            'recon': f"{recon_loss.item():.4f}",
            'reg': f"{reg_loss.item():.4f}",
            'avg': f"{train_loss/train_batches:.4f}"
        })
    
    train_loss /= train_batches
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            batch = batch.to(device)
            reconstruction, embedding, mask = model(batch)
            loss = F.mse_loss(reconstruction[mask], batch[mask])
            val_loss += loss.item()
            val_batches += 1
    
    val_loss /= val_batches
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    epoch_time = time.time() - epoch_start
    
    print(f"📊 Epoch {epoch+1} Results:")
    print(f"  Train Loss: {train_loss:.6f}")
    print(f"  Val Loss: {val_loss:.6f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Time: {epoch_time:.1f}s")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': CONFIG,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'history': history
        }, '/content/best_model.pt')
        print(f"  💾 Best model saved (val_loss: {val_loss:.6f})")
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': CONFIG,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'history': history
        }, f'/content/checkpoint_epoch_{epoch+1}.pt')
        print(f"  📁 Checkpoint saved: epoch_{epoch+1}.pt")

total_time = time.time() - start_time
print(f"🎉 Training completed!")
print(f"⏱️ Total time: {total_time/60:.1f} minutes")
print(f"🏆 Best validation loss: {best_val_loss:.6f}")

## 📈 Results Visualization

In [ ]:
# Plot training results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss', color='blue')
plt.plot(history['val_loss'], label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(history['learning_rate'], color='green')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.semilogy(history['train_loss'], label='Train Loss', color='blue')
plt.semilogy(history['val_loss'], label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Progress (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Save training history
with open('/content/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("📊 Training plots generated and history saved")

## 💾 Save Final Model to Google Drive

In [ ]:
# Save to Google Drive
drive_model_path = '/content/drive/MyDrive/stock_bot_models'
!mkdir -p "{drive_model_path}"

# Create timestamp for model versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'autoencoder_colab_{timestamp}'

# Save model files
model_dir = f'{drive_model_path}/{model_name}'
!mkdir -p "{model_dir}"

print(f"💾 Saving model to: {model_dir}")

# Copy best model and artifacts
!cp /content/best_model.pt "{model_dir}/model.pt"
!cp /content/training_history.json "{model_dir}/training_history.json"
!cp /content/training_results.png "{model_dir}/training_results.png"

# Save model info
model_info = {
    'model_name': model_name,
    'timestamp': timestamp,
    'config': CONFIG,
    'best_val_loss': best_val_loss,
    'total_params': total_params,
    'training_time_minutes': total_time / 60,
    'data_shape': {
        'seq_len': train_dataset.seq_len,
        'num_features': train_dataset.num_features
    }
}

with open(f'{model_dir}/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Model saved to: {model_dir}")
print(f"📁 Files saved:")
print(f"  - model.pt (best checkpoint)")
print(f"  - training_history.json")
print(f"  - training_results.png")
print(f"  - model_info.json")

# Display final summary
print(f"🎉 Training Summary:")
print(f"  Model: {model_name}")
print(f"  Best Val Loss: {best_val_loss:.6f}")
print(f"  Parameters: {total_params:,}")
print(f"  Training Time: {total_time/60:.1f} minutes")
print(f"  Embedding Dim: {CONFIG['embedding_dim']}")
print(f"  Data Shape: {train_dataset.seq_len} x {train_dataset.num_features}")